# DinoV2 Dogs-vs-Cats Hypothesis: Linear Probing vs Partial Unfreeze (Parallel 2xT4)

This notebook is a producer workflow that trains two experiments in sequence:
1. `linear_probe` (freeze backbone, train only classifier head)
2. `baseline_02` (the default setup from `02_dinov2_training_parallel_t4x2.ipynb`)

Flow: bootstrap -> config -> preprocess -> optional sanity -> train linear probe -> train baseline -> compare -> review/export artifacts.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

REPO_URL = 'https://github.com/mruniverse8/kaggle-experiments-.git'
REPO_DIR = Path('/kaggle/working/kaggle-experiments-')
BRANCH = 'dogs_vs_cats_v2'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', '--all'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

current_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
current_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print('Git branch:', current_branch)
print('Git commit:', current_commit)
print('Repo ready at:', REPO_DIR)


In [ ]:
import os
import json
from pathlib import Path

PATHS_CFG = os.environ.get('PATHS_CFG', 'dogs_vs_cats/configs/paths_kaggle.json')
CFG_LINEAR = os.environ.get('CFG_LINEAR', 'dogs_vs_cats/configs/experiments/dinov2_vitb14_linear_probe.json')
CFG_BASELINE = os.environ.get('CFG_BASELINE', 'dogs_vs_cats/configs/experiments/dinov2_vitb14_parallel_t4x2_small.json')
RUN_SANITY = os.environ.get('RUN_SANITY', '0') == '1'

PATHS = json.loads(Path(PATHS_CFG).read_text())
EXPERIMENTS = [
    {'profile': 'linear_probe', 'cfg_path': CFG_LINEAR},
    {'profile': 'baseline_02', 'cfg_path': CFG_BASELINE},
]

print('Using paths config:', PATHS_CFG)
print('Run sanity checks:', RUN_SANITY)
print('Experiments (training order):')
for exp in EXPERIMENTS:
    payload = json.loads(Path(exp['cfg_path']).read_text())
    exp['experiment_name'] = payload['experiment_name']
    exp['config_payload'] = payload
    print('-', exp['profile'], '->', exp['cfg_path'], '| experiment_name=', payload['experiment_name'])

for key in ['train_dir', 'eval_dir', 'test_dir', 'train_zip', 'test_zip', 'sample_submission_csv']:
    value = PATHS.get(key, '')
    if not value:
        print(f"{key}: <empty>")
        continue
    exists = Path(value).exists()
    print(f"{key}: {value} | exists={exists}")


In [ ]:
import sys
import subprocess
from pathlib import Path

manifest_dir = Path(PATHS['manifests_dir'])
required = [
    manifest_dir / 'train_manifest.csv',
    manifest_dir / 'val_manifest.csv',
    manifest_dir / 'test_manifest.csv',
]

if all(path.exists() for path in required):
    print('Preprocess skipped: manifests already exist')
else:
    # Use baseline config for split/preprocess defaults.
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/preprocess_competition_data.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', CFG_BASELINE,
    ], check=True)


In [ ]:
import sys
import subprocess

if RUN_SANITY:
    for exp in EXPERIMENTS:
        print('Running sanity check for:', exp['experiment_name'])
        subprocess.run([
            sys.executable,
            'dogs_vs_cats/src/sanity_check_random_init.py',
            '--paths-config', PATHS_CFG,
            '--experiment-config', exp['cfg_path'],
        ], check=True)
else:
    print('Sanity checks skipped (set RUN_SANITY=1 to enable).')


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

RUNS = []
for exp in EXPERIMENTS:
    print('\n' + '=' * 90)
    print('TRAINING:', exp['profile'], '->', exp['experiment_name'])
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/dinov2_pipeline.py',
        '--mode', 'train',
        '--paths-config', PATHS_CFG,
        '--experiment-config', exp['cfg_path'],
    ], check=True)

    report_path = Path(PATHS['reports_dir']) / f"{exp['experiment_name']}_training_summary.json"
    if not report_path.exists():
        raise FileNotFoundError(f'Training summary missing: {report_path}')
    summary = json.loads(report_path.read_text())
    RUNS.append({
        'profile': exp['profile'],
        'cfg_path': exp['cfg_path'],
        'experiment_name': exp['experiment_name'],
        'summary': summary,
        'report_path': str(report_path),
    })
    print('Final val metrics:', summary.get('final_val_metrics', {}))


In [ ]:
import pandas as pd

rows = []
for run in RUNS:
    metrics = run['summary'].get('final_val_metrics', {})
    rows.append({
        'profile': run['profile'],
        'experiment_name': run['experiment_name'],
        'val_auc': metrics.get('val_auc'),
        'val_logloss': metrics.get('val_logloss'),
        'val_accuracy': metrics.get('val_accuracy'),
        'val_loss': metrics.get('val_loss'),
        'best_epoch': run['summary'].get('best_epoch'),
        'best_monitor': run['summary'].get('best_monitor'),
        'monitor': run['summary'].get('monitor'),
    })

compare_df = pd.DataFrame(rows)
display(compare_df.sort_values(by=['val_auc', 'val_accuracy'], ascending=[False, False]).reset_index(drop=True))

linear_row = compare_df.loc[compare_df['profile'] == 'linear_probe']
base_row = compare_df.loc[compare_df['profile'] == 'baseline_02']
if len(linear_row) == 1 and len(base_row) == 1:
    linear_row = linear_row.iloc[0]
    base_row = base_row.iloc[0]
    deltas = {
        'baseline_minus_linear_val_auc': float(base_row['val_auc'] - linear_row['val_auc']),
        'baseline_minus_linear_val_logloss': float(base_row['val_logloss'] - linear_row['val_logloss']),
        'baseline_minus_linear_val_accuracy': float(base_row['val_accuracy'] - linear_row['val_accuracy']),
    }
    print('Metric deltas (baseline - linear_probe):', deltas)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plot_keys = [
    'train_loss_plot',
    'val_metrics_plot',
    'grad_norm_plot',
    'val_confusion_matrix_plot',
]

for run in RUNS:
    print('\nPLOTS:', run['experiment_name'])
    files = run['summary'].get('files', {})
    for key in plot_keys:
        path = files.get(key, '')
        if not path:
            continue
        p = Path(path)
        if not p.exists():
            continue
        plt.figure(figsize=(8, 4))
        plt.imshow(mpimg.imread(p))
        plt.title(f"{run['profile']} | {p.name}")
        plt.axis('off')
        plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

run_tables = []
for run in RUNS:
    files = run['summary'].get('files', {})
    train_hist = pd.read_csv(files['train_history_csv']) if Path(files.get('train_history_csv', '')).exists() else pd.DataFrame()
    eval_hist = pd.read_csv(files['eval_history_csv']) if Path(files.get('eval_history_csv', '')).exists() else pd.DataFrame()
    metrics = run['summary'].get('final_val_metrics', {})
    run_tables.append({
        'run': run,
        'train_hist': train_hist,
        'eval_hist': eval_hist,
        'metrics': metrics,
    })

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
ax_train, ax_auc = axes[0]
ax_logloss, ax_summary = axes[1]

for row in run_tables:
    name = row['run']['profile']
    train_hist = row['train_hist']
    eval_hist = row['eval_hist']
    if not train_hist.empty:
        ax_train.plot(train_hist['global_step'], train_hist['train_loss'], label=name, linewidth=2)
    if not eval_hist.empty:
        ax_auc.plot(eval_hist['global_step'], eval_hist['val_auc'], label=name, marker='o', linewidth=2)
        ax_logloss.plot(eval_hist['global_step'], eval_hist['val_logloss'], label=name, marker='o', linewidth=2)

ax_train.set_title('Train Loss vs Global Step')
ax_train.set_xlabel('Global step')
ax_train.set_ylabel('train_loss')
ax_train.grid(alpha=0.25)
ax_train.legend()

ax_auc.set_title('Validation AUC vs Global Step')
ax_auc.set_xlabel('Global step')
ax_auc.set_ylabel('val_auc')
ax_auc.grid(alpha=0.25)
ax_auc.legend()

ax_logloss.set_title('Validation Logloss vs Global Step')
ax_logloss.set_xlabel('Global step')
ax_logloss.set_ylabel('val_logloss')
ax_logloss.grid(alpha=0.25)
ax_logloss.legend()

profiles = [row['run']['profile'] for row in run_tables]
x = np.arange(len(profiles))
auc_vals = [row['metrics'].get('val_auc', np.nan) for row in run_tables]
acc_vals = [row['metrics'].get('val_accuracy', np.nan) for row in run_tables]
logloss_vals = [row['metrics'].get('val_logloss', np.nan) for row in run_tables]

bar_w = 0.34
bars_auc = ax_summary.bar(x - bar_w / 2, auc_vals, width=bar_w, label='val_auc')
bars_acc = ax_summary.bar(x + bar_w / 2, acc_vals, width=bar_w, label='val_accuracy')
ax_summary.set_ylim(0.0, 1.05)
ax_summary.set_xticks(x, profiles)
ax_summary.set_ylabel('AUC / Accuracy')
ax_summary.set_title('Final Validation Metrics (Single Comparison Panel)')
ax_summary.grid(alpha=0.2, axis='y')

ax_summary_2 = ax_summary.twinx()
ax_summary_2.plot(x, logloss_vals, color='tab:red', marker='D', linewidth=2, label='val_logloss')
ax_summary_2.set_ylabel('Logloss (lower is better)')

for bar in list(bars_auc) + list(bars_acc):
    height = bar.get_height()
    if np.isfinite(height):
        ax_summary.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f"{height:.4f}", ha='center', va='bottom', fontsize=9)
for xi, val in enumerate(logloss_vals):
    if np.isfinite(val):
        ax_summary_2.text(xi, val, f"{val:.4f}", color='tab:red', fontsize=9, ha='left', va='bottom')

handles_1, labels_1 = ax_summary.get_legend_handles_labels()
handles_2, labels_2 = ax_summary_2.get_legend_handles_labels()
ax_summary.legend(handles_1 + handles_2, labels_1 + labels_2, loc='best')

plt.suptitle('Linear Probe vs Baseline (02) - Unified Training and Validation Comparison', fontsize=14, y=0.98)
plt.tight_layout()

special_plot_path = Path(PATHS['plots_dir']) / 'linear_probe_vs_baseline02_special_comparison.png'
special_plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(special_plot_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved special comparison plot to:', special_plot_path)


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

EXPORTS = []
for run in RUNS:
    print('\nEXPORTING:', run['experiment_name'])
    output = subprocess.check_output([
        sys.executable,
        'dogs_vs_cats/src/export_experiment_artifacts.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', run['cfg_path'],
    ]).decode()
    info = json.loads(output)
    EXPORTS.append(info)
    print('Exported dir:', info['exported_dir'])
    print('Manifest:', info['manifest_path'])
    print('Bundle index:', info['bundle_index_path'])


In [ ]:
from pathlib import Path

for run in RUNS:
    summary = run['summary']
    print('\n' + '=' * 90)
    print('PROFILE:', run['profile'])
    print('EXPERIMENT:', run['experiment_name'])
    print('MONITOR:', summary.get('monitor', ''))
    print('BEST MONITOR:', summary.get('best_monitor'))
    print('BEST EPOCH:', summary.get('best_epoch'))
    files = summary.get('files', {})
    for key in sorted(files.keys()):
        p = Path(files[key])
        print('-', key, '->', p, '| exists=', p.exists())
